# 01 – Model exploration: the iML1515 network and a first FBA

**Research question.** What does a genome-scale metabolic model of *E. coli* K-12 contain, and what growth rate does flux balance analysis (FBA) predict for it?

**Biological motivation.** Every essentiality call later in the project is a statement about this network: which reactions exist, which genes are needed for each of them, and which nutrients can enter the cell. Knowing what the model represents, and what it leaves out, is the basis for judging its predictions.

**What this notebook does.**
- loads the iML1515 model (Monk et al. 2017) and counts its genes, reactions and metabolites
- looks at one reaction, its flux bounds and its gene-protein-reaction (GPR) rule
- inspects the default growth medium, defined by the open exchange reactions
- runs a first FBA and compares the predicted growth rate under a few conditions

**Expected outputs.** Model statistics (1,516 gene entries including the placeholder `s0001`, 2,712 reactions, 1,877 metabolites), the 24 open exchange reactions of the default medium, and a wild-type growth rate of about 0.877 h⁻¹ on aerobic glucose minimal medium. Nothing is written to `results/`.

Run the cells top to bottom. The first run downloads the model from BiGG Models into `data/` (internet required); later runs load it from disk.

In [1]:
import cobra
import pandas as pd
from cobra.io import load_json_model

print("COBRApy version:", cobra.__version__)

COBRApy version: 0.32.1


## 1. Load the model

In [2]:
from pathlib import Path
from urllib.request import urlretrieve

# cobra's load_model("iML1515") currently fails: BiGG redirects http -> https
# and cobra's downloader does not follow the redirect.
# So we download the JSON once over https and load it from disk.
MODEL_URL = "https://bigg.ucsd.edu/static/models/iML1515.json"
model_path = Path("../data/iML1515.json")
if not model_path.exists():
    urlretrieve(MODEL_URL, model_path)

model = load_json_model(model_path)

print("Genes:      ", len(model.genes))
print("Reactions:  ", len(model.reactions))
print("Metabolites:", len(model.metabolites))

Genes:       1516
Reactions:   2712
Metabolites: 1877


## 2. Look at the objects
Each reaction has bounds and a gene-reaction rule (GPR). `and` = complex (all genes needed), `or` = isozymes (any gene is enough).

The GPR rules link genes to metabolism: deleting one subunit of a complex blocks the reaction, while deleting one of several isozymes leaves it running. Notebook 02 uses exactly this logic for the knockout screen.

In [3]:
r = model.reactions[100]
print(r.id, "|", r.name)
print("Equation:", r.reaction)
print("Bounds:  ", r.bounds)
print("GPR:     ", r.gene_reaction_rule)

g = model.genes[0]
print("\nGene:", g.id, "| reactions:", [x.id for x in g.reactions])

UPPDC1 | Uroporphyrinogen decarboxylase (uroporphyrinogen III)
Equation: 4.0 h_c + uppg3_c --> 4.0 co2_c + cpppg3_c
Bounds:   (0.0, 1000.0)
GPR:      b3997

Gene: b2551 | reactions: ['GHMT2r', 'THFAT', 'ALATA_D2', 'THRA', 'ALATA_L2', 'THRA2']


In [4]:
# How many reactions have a gene rule with 'and' / 'or'?
rules = pd.Series({rx.id: rx.gene_reaction_rule for rx in model.reactions})
print("No GPR:", (rules == "").sum())
print("With 'and':", rules.str.contains(" and ").sum())
print("With 'or': ", rules.str.contains(" or ").sum())

No GPR: 446
With 'and': 313
With 'or':  743


## 3. The default medium
Exchange reactions define what the cell can take up. Negative lower bound = uptake allowed.

The set of open exchange reactions *is* the simulated growth medium, with uptake limits in mmol per gram dry weight per hour (mmol/gDW/h). Which genes are essential depends on it: a gene that only makes a nutrient is dispensable when that nutrient can be taken up.

In [5]:
medium = pd.Series(model.medium).sort_values(ascending=False)
print(len(medium), "open exchange reactions")
medium

24 open exchange reactions


EX_pi_e         1000.0
EX_co2_e        1000.0
EX_fe3_e        1000.0
EX_h_e          1000.0
EX_mn2_e        1000.0
EX_fe2_e        1000.0
EX_zn2_e        1000.0
EX_mg2_e        1000.0
EX_so4_e        1000.0
EX_ca2_e        1000.0
EX_ni2_e        1000.0
EX_cu2_e        1000.0
EX_sel_e        1000.0
EX_cobalt2_e    1000.0
EX_h2o_e        1000.0
EX_mobd_e       1000.0
EX_cl_e         1000.0
EX_nh4_e        1000.0
EX_k_e          1000.0
EX_na1_e        1000.0
EX_tungs_e      1000.0
EX_o2_e         1000.0
EX_slnt_e       1000.0
EX_glc__D_e       10.0
dtype: float64

In [6]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.004565,0,0.00%
cl_e,EX_cl_e,0.004565,0,0.00%
cobalt2_e,EX_cobalt2_e,2.192E-05,0,0.00%
cu2_e,EX_cu2_e,0.0006218,0,0.00%
fe2_e,EX_fe2_e,0.01409,0,0.00%
glc__D_e,EX_glc__D_e,10,6,100.00%
k_e,EX_k_e,0.1712,0,0.00%
mg2_e,EX_mg2_e,0.007608,0,0.00%
mn2_e,EX_mn2_e,0.000606,0,0.00%
mobd_e,EX_mobd_e,6.139E-06,0,0.00%


## 4. First FBA
The objective is the biomass reaction. `slim_optimize()` returns only the growth rate (faster); `optimize()` returns the full flux distribution.

FBA assumes a **steady state**: every internal metabolite is produced exactly as fast as it is consumed. Within that constraint and the flux bounds, a linear program finds the flux distribution that maximises the **biomass reaction**, a pseudo-reaction that drains amino acids, nucleotides, lipids, cofactors and other cell components in fixed proportions. Its flux is read as the growth rate (h⁻¹). Enzyme levels, kinetics and gene regulation are not part of the calculation.

In [7]:
print("Objective:", model.objective.expression)

solution = model.optimize()
print("Growth rate (1/h):", solution.objective_value)
solution.fluxes.sort_values(key=abs, ascending=False).head(10)

Objective: 1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685
Growth rate (1/h): 0.8769972144269716


ATPS4rpp      70.432494
H2Otpp       -47.162365
H2Otex       -47.162365
EX_h2o_e      47.162365
CYTBO3_4pp    44.256288
NADH16pp      37.997044
CO2tex       -24.003293
EX_co2_e      24.003293
CO2tpp       -24.003293
O2tex         22.131763
Name: fluxes, dtype: float64

**Questions and answers**

1. *What is the default carbon source and its uptake limit?*
   D-glucose (`EX_glc__D_e`), the only organic carbon source in the default medium, with a maximum uptake of 10 mmol/gDW/h (lower bound −10).

2. *Is oxygen available by default?*
   Yes. `EX_o2_e` is open with a bound of 1000 mmol/gDW/h, which is effectively unlimited, so the default condition is aerobic. Closing it (section 5) lowers the predicted growth rate from 0.877 to 0.158 h⁻¹, about 5.6-fold, because fermentation yields far less ATP per glucose than respiration.

3. *How does the computed growth rate compare with literature values for E. coli on glucose minimal medium?*
   The model predicts 0.877 h⁻¹. LaCroix et al. (2015) measured 0.69 ± 0.02 h⁻¹ for MG1655 in glucose M9 medium at 37 °C at the start of their evolution experiments, 0.82 ± 0.04 h⁻¹ after physiological adaptation by repeated passaging in exponential phase, and about 1.0 to 1.1 h⁻¹ for the evolved populations. The prediction is therefore about 27 % above the unadapted wild type and lies between the adapted wild type and the evolved strains. This is expected: FBA returns the maximum growth that the stoichiometry allows for an assumed glucose uptake of 10 mmol/gDW/h, while a real cell may take up glucose at a different rate and is not necessarily growing at its stoichiometric optimum.

   Reference: LaCroix R.A. et al. (2015) Use of adaptive laboratory evolution to discover key mutations enabling rapid growth of *Escherichia coli* K-12 MG1655 on glucose minimal medium. *Appl Environ Microbiol* 81:17–30. https://doi.org/10.1128/AEM.02246-14

## 5. Growth under different conditions
Use `with model:` so changes are reverted automatically after the block.

Each block below is an in-silico experiment: closing or opening one exchange reaction changes the medium, and FBA recomputes the maximal growth rate.

In [8]:
def growth(model):
    return model.slim_optimize(error_value=0.0)

results = {}

results["default"] = growth(model)

with model:
    model.reactions.EX_o2_e.lower_bound = 0       # no oxygen
    results["anaerobic"] = growth(model)

with model:
    model.reactions.EX_glc__D_e.lower_bound = 0   # remove glucose
    model.reactions.EX_ac_e.lower_bound = -10     # acetate instead
    results["acetate"] = growth(model)

with model:
    model.reactions.EX_glc__D_e.lower_bound = 0
    model.reactions.EX_glyc_e.lower_bound = -10   # glycerol instead
    results["glycerol"] = growth(model)

pd.Series(results, name="growth rate (1/h)")

default      0.876997
anaerobic    0.157540
acetate      0.210099
glycerol     0.494713
Name: growth rate (1/h), dtype: float64

## Notes

**Observations**
- The model lists 1,516 gene entries, but one of them, `s0001`, is a placeholder for spontaneous reactions. It has to be excluded from the knockout screen.
- 446 of the 2,712 reactions have no gene rule; most of them (337) are exchange, demand and sink reactions at the model boundary.
- The carbon sources in section 5 were compared at the same molar uptake (10 mmol/gDW/h), so glucose supplies 60, glycerol 30 and acetate 20 mmol carbon per gDW per hour. The growth rates follow the same order: 0.877, 0.495 and 0.210 h⁻¹.

**To check next**
- Which genes become essential when a gene is deleted, and does the answer depend on the medium?

Next notebook: single-gene knockout screen.